# Real or Fake Image Recognition Task Using a Convolutional Neural Network

I built a model that receives a colored image and predicts whether it is **real (0)** or **generated (1)**. This notebook includes the complete code and the actual results from running it using `CIFAR10's "real"` images and a `diffused model's "fake"` images.

# 1. The task and approach

The supplied images on the two folders are 32 pixels wide and 32 pixels high wherein each pixel has red, green, and blue values. I used a **convolutional neural network (CNN)** which is the most appropriate model for this machine learning task as it elarns small image patterns and combines them to make a decision.

I used a small model to balance the image-classification task wherein the model proddudces only `two scores` in which the larger score determines its actual prediction.

# 2. Setup of the model

| Setting | Value | 
| --- | --- | 
| Maximum development epochs | 50 |
| Batch size | 64 |
| Starting learning rate | 0.001 |
| Weight penalty | 0.01 |
| Seed | 42 |
| CPU threads | 4 |

These are the chosen starting settings with the help of an LLM's suggestion. For reference, these are the suggested values: https://chatgpt.com/share/6acf7100-ab18-83ec-8341-f8193559edca.

In [ ]:
import csv
import json
import math
import random
import time
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch import nn
from IPython.display import Markdown, display

roots = [Path.cwd(), *Path.cwd().parents]
candidates = [root / 'part2' for root in roots] + roots
PART_FOLDER = next((folder for folder in candidates if (folder / 'train_v1' / 'cifar').is_dir()), Path.cwd())
DATA_DIR = PART_FOLDER / 'train_v1'
OUTPUT_DIR = PART_FOLDER / 'simplified_learning' / 'student_submission_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

settings = SimpleNamespace(epochs=40, batch_size=128, learning_rate=0.001, seed=42, threads=4)
torch.set_num_threads(settings.threads)
torch.use_deterministic_algorithms(True)
workflow_start = time.perf_counter()
print('PyTorch:', torch.__version__, '| Device: CPU')
print('Image folder:', DATA_DIR)
print('Output folder:', OUTPUT_DIR)

plt.rcParams.update({
    'figure.dpi': 130, 'savefig.dpi': 160, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.labelcolor': '#334155', 'text.color': '#172554',
    'axes.prop_cycle': plt.cycler(color=['#0f766e', '#c2410c', '#1d4ed8', '#7c3aed']),
})
